# Forest Fire Detection — Part 3: Evaluation

Evaluates the trained **FireNet-CNN** (from `02_model_training.ipynb`) on the **test set**, the 150 images never used during training or tuning.

## 0. Data Preparation

Same code as `01_preprocessing.ipynb`, so the **test set is identical** across all notebooks.

In [ ]:
!pip install -q kaggle scikit-learn seaborn

import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              precision_score, recall_score, f1_score,
                              roc_curve, auc, accuracy_score)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Download the FIRE Dataset
!kaggle datasets download -d phylake1337/fire-dataset -p /content/data --unzip

DATA_DIR = "/content/data/fire_dataset"
print(os.listdir(DATA_DIR))

In [ ]:
fire_dir = os.path.join(DATA_DIR, "fire_images")
non_fire_dir = os.path.join(DATA_DIR, "non_fire_images")

In [ ]:
IMG_SIZE = 128

def load_images(folder, label, img_size=IMG_SIZE):
    images, labels = [], []
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        path = os.path.join(folder, fname)
        try:
            img = tf.keras.utils.load_img(path, target_size=(img_size, img_size))
            img_array = tf.keras.utils.img_to_array(img) / 255.0   # normalize 0-1
            images.append(img_array)
            labels.append(label)
        except Exception as e:
            print(f"Skipping corrupt file: {fname} ({e})")
    return images, labels

print("Loading fire images...")
fire_imgs, fire_labels = load_images(fire_dir, label=1)

print("Loading non-fire images...")
non_fire_imgs, non_fire_labels = load_images(non_fire_dir, label=0)

X = np.array(fire_imgs + non_fire_imgs, dtype=np.float32)
y = np.array(fire_labels + non_fire_labels, dtype=np.float32)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

In [ ]:
# Split: 70% train, 15% val, 15% test  (stratified to preserve class balance)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape[0]} images  ({y_train.sum():.0f} fire / {(y_train==0).sum():.0f} non-fire)")
print(f"Val:   {X_val.shape[0]} images  ({y_val.sum():.0f} fire / {(y_val==0).sum():.0f} non-fire)")
print(f"Test:  {X_test.shape[0]} images  ({y_test.sum():.0f} fire / {(y_test==0).sum():.0f} non-fire)")

In [ ]:
train_datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1
)

# No augmentation for validation/test — only real, unmodified images are evaluated
val_datagen = ImageDataGenerator()
test_datagen = ImageDataGenerator()

BATCH_SIZE = 32

train_generator = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE, shuffle=True)
val_generator = val_datagen.flow(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)
test_generator = test_datagen.flow(X_test, y_test, batch_size=BATCH_SIZE, shuffle=False)

## 1. Load the Trained Model

Upload `firenet_cnn_final.keras` to the Colab Files panel (📁 on the left) before running this cell. No retraining is done here.

In [ ]:
model = tf.keras.models.load_model('firenet_cnn_final.keras')
print(f"Loaded model with {model.count_params():,} parameters")

## 2. Test Metrics

Threshold 0.5: probability ≥ 0.5 → Fire. Recall is the key metric (a missed fire is the costliest error).

In [ ]:
# Get predictions on the test set
y_pred_probs = model.predict(X_test).ravel()
y_pred = (y_pred_probs >= 0.5).astype(int)
y_true = y_test.astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Test Accuracy:  {acc:.4f}")
print(f"Precision:      {prec:.4f}")
print(f"Recall:         {rec:.4f}")
print(f"F1-Score:       {f1:.4f}")
print()
print(classification_report(y_true, y_pred, target_names=['No Fire', 'Fire']))

**Analysis:** _(write 2–3 short points: accuracy, precision, recall, F1, and what they mean)_

## 3. Confusion Matrix

Shows exactly where the model is right and wrong: fires caught, fires missed, false alarms.

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=['No Fire', 'Fire'],
            yticklabels=['No Fire', 'Fire'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix — FireNet-CNN')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Positives  (correctly caught fire):     {tp}")
print(f"True Negatives  (correctly caught no-fire):   {tn}")
print(f"False Positives (false alarms):               {fp}")
print(f"False Negatives (missed fires — most costly): {fn}")

**Analysis:** _(how many fires were missed vs false alarms, and which matters more)_

## 4. ROC Curve and AUC

AUC = how well the model separates fire from non-fire across all thresholds (1.0 = perfect, 0.5 = random guess).

In [ ]:
# ROC Curve & AUC
fpr, tpr, thresholds = roc_curve(y_true, y_pred_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve — FireNet-CNN')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"ROC-AUC: {roc_auc:.4f}")

**Analysis:** _(what the AUC value says about the model)_

## 5. Sample Predictions

15 random test images. Green = correct, red = wrong.

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
sample_idx = np.random.choice(len(X_test), 15, replace=False)

for i, idx in enumerate(sample_idx):
    ax = axes[i // 5, i % 5]
    ax.imshow(X_test[idx])
    true_label = "Fire" if y_true[idx] == 1 else "No Fire"
    pred_label = "Fire" if y_pred[idx] == 1 else "No Fire"
    prob = y_pred_probs[idx]
    color = "green" if true_label == pred_label else "red"
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({prob:.2f})", color=color, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Threshold Analysis (Extension)

Lowering the threshold below 0.5 makes the model say "Fire" more easily → it should catch more fires (higher recall) but raise more false alarms (lower precision).

In [ ]:
import pandas as pd

rows = []
for t in [0.5, 0.4, 0.3, 0.2]:
    pred_t = (y_pred_probs >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_true, pred_t).ravel()
    rows.append({
        "Threshold": t,
        "Recall": round(recall_score(y_true, pred_t), 4),
        "Precision": round(precision_score(y_true, pred_t), 4),
        "F1": round(f1_score(y_true, pred_t), 4),
        "Missed fires (FN)": int(fn_t),
        "False alarms (FP)": int(fp_t),
    })
threshold_df = pd.DataFrame(rows)
display(threshold_df)

**Analysis:** _(does a lower threshold catch more fires? how many extra false alarms? which threshold would you choose and why?)_

## 7. Misclassified Images

Every test image the model got wrong, to understand why it failed.

In [ ]:
wrong = np.where(y_true != y_pred)[0]
print(f"{len(wrong)} misclassified test images")

if len(wrong) > 0:
    n = len(wrong)
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.6), squeeze=False)
    for ax, idx in zip(axes[0], wrong):
        ax.imshow(X_test[idx])
        true_label = "Fire" if y_true[idx] == 1 else "No Fire"
        pred_label = "Fire" if y_pred[idx] == 1 else "No Fire"
        ax.set_title(f"True: {true_label}\nPred: {pred_label} ({y_pred_probs[idx]:.2f})",
                     color='red', fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

**Analysis:** _(look at each image: why might the model have been confused? e.g. orange sunset, small or hidden fire, smoke without flames)_